NN1——area1

In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

/bin/bash: line 1: nvidia-smi: command not found


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import scipy.io
from pathlib import Path
import os
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader, random_split
import numpy as np
from scipy.io import loadmat
import pandas as pd
import h5py

In [3]:
"""
#Load data
from google.colab import files
uploaded = files.upload()
"""


#Connect to the cloud
from google.colab import drive
drive.mount('/content/drive')


# 直接从指定路径读取 .mat 数据文件
file_path = '/content/drive/MyDrive/Colab_Notebooks/NN_solve_MM_Field_V2/data/data.mat'  # 根据实际文件名和路径更改

"""
# 加载 .mat 文件
data = scipy.io.loadmat(file_path)
"""


# 加载 .mat 文件
with h5py.File(file_path, 'r') as file:
    data = {key: file[key][:] for key in file.keys()}


Mounted at /content/drive


In [4]:
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    # 保证每次运行结果一样
    # 在神经网络中，参数默认是进行随机初始化的。
    #如果不设置的话每次训练时的初始化都是随机的，导致结果不确定。
    #如果设置初始化，则每次初始化都是固定的。

# 设置随机数种子  设定固定的随机数以便结果可以复现
setup_seed(888888)

In [5]:
"""
# Load training data (boundary points), residual and interface points from .mat file
# All points are generated in Matlab
main_folder_path = Path('.')
if not os.path.exists(main_folder_path / 'Contour_Plots_NL'):
    os.makedirs(main_folder_path / 'Contour_Plots_NL')

data = scipy.io.loadmat(main_folder_path / 'data.mat')
"""

R11=data['R111']

R11=R11.T #因为使用h5py，所以需要将此矩阵转置

L = R11[:,1:44]
R = R11[:,44]

#重塑数组R
R= R.reshape(-1, 1)

# 标准化L
scaler_L = StandardScaler()
L_normalized = scaler_L.fit_transform(L)  # 标准化L

# 列标准化R
scaler_R = StandardScaler()
R_normalized = scaler_R.fit_transform(R)  # 标准化R的每一列

# 转换回torch张量
L_tensor = torch.tensor(L_normalized, dtype=torch.float32)
R_tensor = torch.tensor(R_normalized, dtype=torch.float32)

# 计算数据集大小
dataset_size = len(L_tensor)
train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size  # 确保总和为整个数据集大小

# 创建TensorDataset
full_dataset = TensorDataset(L_tensor, R_tensor)

# 随机划分数据集
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

In [6]:
print(R)

[[ 0.24087525]
 [ 0.06488521]
 [-0.14642194]
 ...
 [-0.16153131]
 [-0.09235263]
 [ 0.0403506 ]]


In [7]:
#Neural Network
#The neural network has 2 inputs, 1 output, and 3 hidden layers with 32 neurons each
#MPL (Multilayer Perceptron) is the most fundamental fully connected layer neural network
class MLP1(torch.nn.Module):
  def __init__(self):
    super(MLP1,self).__init__()
    self.net=torch.nn.Sequential(
     torch.nn.Linear(43,100),
     torch.nn.LeakyReLU(0.01),
     torch.nn.Linear(100,100),
     torch.nn.LeakyReLU(0.01),
     torch.nn.Linear(100,100),
     torch.nn.LeakyReLU(0.01),
     torch.nn.Linear(100,100),
     torch.nn.LeakyReLU(0.01),
     torch.nn.Linear(100,1),
    )

  def forward(self,x):
      return self.net(x)

In [8]:
# 假设MLP1和其它所需库已经被正确导入
model = MLP1()

# 确保模型和数据在相同的设备上
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 数据加载器
train_loader = DataLoader(train_dataset, batch_size=50000, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=500000, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=500000, shuffle=False)

epochs = 500
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        # 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        # 后向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_loader.dataset)

    # 验证过程
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # 定期保存模型和优化器状态
    if (epoch + 1) % 10 == 0:  # 每100个epoch保存一次
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }
        torch.save(checkpoint, f'//content/drive/MyDrive/Colab_Notebooks/NN_solve_MM_Field_V2/Weigh/model_epoch_{epoch + 1}.pth')

# 最后保存模型和优化器状态
checkpoint = {
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_loss,
    'val_loss': val_loss,
}
torch.save(checkpoint, '/content/drive/MyDrive/Colab_Notebooks/NN_solve_MM_Field_V2/Weigh/model_final_2.pth')

Epoch 1/500, Train Loss: 0.9705, Val Loss: 0.8926
Epoch 2/500, Train Loss: 0.8252, Val Loss: 0.7842
Epoch 3/500, Train Loss: 0.7658, Val Loss: 0.7513
Epoch 4/500, Train Loss: 0.7213, Val Loss: 0.6942
Epoch 5/500, Train Loss: 0.6604, Val Loss: 0.6419
Epoch 6/500, Train Loss: 0.6222, Val Loss: 0.6121
Epoch 7/500, Train Loss: 0.5924, Val Loss: 0.5791
Epoch 8/500, Train Loss: 0.5588, Val Loss: 0.5452
Epoch 9/500, Train Loss: 0.5171, Val Loss: 0.4992
Epoch 10/500, Train Loss: 0.4767, Val Loss: 0.4661
Epoch 11/500, Train Loss: 0.4464, Val Loss: 0.4405
Epoch 12/500, Train Loss: 0.4186, Val Loss: 0.4123
Epoch 13/500, Train Loss: 0.3915, Val Loss: 0.3858
Epoch 14/500, Train Loss: 0.3706, Val Loss: 0.3694
Epoch 15/500, Train Loss: 0.3505, Val Loss: 0.3457
Epoch 16/500, Train Loss: 0.3344, Val Loss: 0.3334
Epoch 17/500, Train Loss: 0.3173, Val Loss: 0.3168
Epoch 18/500, Train Loss: 0.3042, Val Loss: 0.3192
Epoch 19/500, Train Loss: 0.2936, Val Loss: 0.2889
Epoch 20/500, Train Loss: 0.2769, Val Lo

KeyboardInterrupt: 

In [9]:
# 测试过程
model.eval()  # 确保模型处于评估模式
test_loss = 0.0
with torch.no_grad():  # 不计算梯度，以节省计算资源和内存
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        test_loss += loss.item() * inputs.size(0)
test_loss /= len(test_loader.dataset)

print(f"Test Loss: {test_loss:.4f}")

Test Loss: 0.0895


In [19]:
#Load data
from google.colab import files
uploaded = files.upload()

# Load training data (boundary points), residual and interface points from .mat file
# All points are generated in Matlab

"""
main_folder_path = Path('.')
if not os.path.exists(main_folder_path / 'Contour_Plots_NL'):
    os.makedirs(main_folder_path / 'Contour_Plots_NL')
"""

# 假设上传的文件名是 'data1.mat'
file_path = 'data1.mat'

# 加载 .mat 文件
with h5py.File(file_path, 'r') as file:
    data1 = {key: file[key][:] for key in file.keys()}

L_pre=data1['Pred']
L_pre=L_pre.T #因为使用h5py，所以需要将此矩阵转置


# 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




# 使用之前训练数据上 fit 过的 scaler_L 进行变换
L_pre_normalized = scaler_L.transform(L_pre)

# 转换成张量
L_pre_tensor = torch.tensor(L_pre_normalized, dtype=torch.float32)

# 确保模型在正确的设备上
model.to(device)

# 确保输入数据也在同一设备上
L_pre_tensor = L_pre_tensor.to(device)



# 使用模型进行预测
model.eval()  # 确保模型处于评估模式
R_pred = model(L_pre_tensor)




Saving data1.mat to data1.mat


In [20]:
print(R_pred)

tensor([[1.2437],
        [1.7482],
        [1.7743],
        [2.4954],
        [2.1184],
        [2.2013],
        [1.9467],
        [1.5618],
        [1.1504],
        [1.1243],
        [1.6153],
        [1.8020],
        [2.4166],
        [2.1335],
        [2.2230],
        [1.9426],
        [1.6675],
        [1.1470],
        [1.1225],
        [1.6904],
        [1.9395],
        [2.2807],
        [2.1856],
        [2.1527],
        [2.1094],
        [1.9809],
        [1.3447],
        [1.4239],
        [1.8891],
        [2.0915],
        [2.3099],
        [2.4747],
        [2.3057],
        [2.2482],
        [2.1240],
        [1.6435],
        [1.5500],
        [2.2030],
        [2.3193],
        [2.5743],
        [2.7213],
        [2.4215],
        [2.3768],
        [2.2555],
        [1.9403],
        [1.7993],
        [2.5823],
        [2.7326],
        [2.9562],
        [2.9287],
        [2.5681],
        [2.4851],
        [2.3665],
        [2.1424],
        [2.2689],
        [2

In [21]:
#Download the value of R_pred
#Save xy
import numpy as np
from google.colab import files

# 将预测的结果从 GPU 移到 CPU
R_pred_cpu = R_pred.detach().cpu()

# 现在可以安全地将张量转换为 NumPy 数组
R_pred_numpy = R_pred_cpu.numpy()


# 使用scaler_R的inverse_transform方法还原标准化前的数据
R_pred_original = scaler_R.inverse_transform(R_pred_numpy)

np.savetxt('R_pred_original.csv', R_pred_original, delimiter=',')
files.download('R_pred_original.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

重新训练之前的网络

In [ ]:
# 定义保存模型的路径，确保路径中的文件夹已经存在
model_weight_path = '/content/drive/MyDrive/data/model_final.pth'

# 假设MLP1和其它所需库已经被正确导入
model = MLP1()

# 加载模型权重
if os.path.exists(model_weight_path):
    model.load_state_dict(torch.load(model_weight_path))
    print("Model weights loaded successfully.")

# 确保模型和数据在相同的设备上
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Model weights loaded successfully.


MLP1(
  (net): Sequential(
    (0): Linear(in_features=43, out_features=200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=200, out_features=300, bias=True)
    (3): ReLU()
    (4): Linear(in_features=300, out_features=200, bias=True)
    (5): ReLU()
    (6): Linear(in_features=200, out_features=1, bias=True)
  )
)

NN2——area1

In [ ]:
#Load data
from google.colab import files
uploaded = files.upload()


#Connect to the cloud
from google.colab import drive
drive.mount('/content/drive')

Saving data.mat to data.mat
Saving data1.mat to data1.mat
Mounted at /content/drive


In [ ]:
def setup_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    # 保证每次运行结果一样
    # 在神经网络中，参数默认是进行随机初始化的。
    #如果不设置的话每次训练时的初始化都是随机的，导致结果不确定。
    #如果设置初始化，则每次初始化都是固定的。

# 设置随机数种子  设定固定的随机数以便结果可以复现
setup_seed(888888)

In [ ]:
# Load training data (boundary points), residual and interface points from .mat file
# All points are generated in Matlab
main_folder_path = Path('.')
if not os.path.exists(main_folder_path / 'Contour_Plots_NL'):
    os.makedirs(main_folder_path / 'Contour_Plots_NL')

data = scipy.io.loadmat(main_folder_path / 'data.mat')

R11=data['R11']
L = R11[:,:11]
R = R11[:,11]

#重塑数组R
R= R.reshape(-1, 1)

# 标准化L
scaler_L = StandardScaler()
L_normalized = scaler_L.fit_transform(L)  # 标准化L

# 列标准化R
scaler_R = StandardScaler()
R_normalized = scaler_R.fit_transform(R)  # 标准化R的每一列

# 转换回torch张量
L_tensor = torch.tensor(L_normalized, dtype=torch.float32)
R_tensor = torch.tensor(R_normalized, dtype=torch.float32)

# 计算数据集大小
dataset_size = len(L_tensor)
train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size  # 确保总和为整个数据集大小

# 创建TensorDataset
full_dataset = TensorDataset(L_tensor, R_tensor)

# 随机划分数据集
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

In [ ]:
#Neural Network
#The neural network has 2 inputs, 1 output, and 3 hidden layers with 32 neurons each
#MPL (Multilayer Perceptron) is the most fundamental fully connected layer neural network
class MLP2(torch.nn.Module):
  def __init__(self):
    super(MLP2,self).__init__()
    self.net=torch.nn.Sequential(
     torch.nn.Linear(11,100),
     torch.nn.ReLU(),
     torch.nn.Linear(100,200),
     torch.nn.ReLU(),
     torch.nn.Linear(200,100),
     torch.nn.ReLU(),
     torch.nn.Linear(100,1),
    )

  def forward(self,x):
      return self.net(x)

In [ ]:
# 假设MLP2和其它所需库已经被正确导入
model2 = MLP2()

# 确保模型和数据在相同的设备上
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model2.to(device)

# 损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(model2.parameters(), lr=0.00001)

# 数据加载器
train_loader = DataLoader(train_dataset, batch_size=10000, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=10000, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=10000, shuffle=False)

epochs = 5000
for epoch in range(epochs):
    model2.train()
    train_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        # 前向传播
        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        # 后向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
    train_loss /= len(train_loader.dataset)

    # 验证过程
    model2.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model2(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

Epoch 1/5000, Train Loss: 0.9929, Val Loss: 0.9570
Epoch 2/5000, Train Loss: 0.9163, Val Loss: 0.8762
Epoch 3/5000, Train Loss: 0.8283, Val Loss: 0.7811
Epoch 4/5000, Train Loss: 0.7287, Val Loss: 0.6786
Epoch 5/5000, Train Loss: 0.6266, Val Loss: 0.5798
Epoch 6/5000, Train Loss: 0.5339, Val Loss: 0.4954
Epoch 7/5000, Train Loss: 0.4594, Val Loss: 0.4316
Epoch 8/5000, Train Loss: 0.4040, Val Loss: 0.3834
Epoch 9/5000, Train Loss: 0.3600, Val Loss: 0.3420
Epoch 10/5000, Train Loss: 0.3203, Val Loss: 0.3034
Epoch 11/5000, Train Loss: 0.2826, Val Loss: 0.2660
Epoch 12/5000, Train Loss: 0.2460, Val Loss: 0.2300
Epoch 13/5000, Train Loss: 0.2110, Val Loss: 0.1959
Epoch 14/5000, Train Loss: 0.1788, Val Loss: 0.1652
Epoch 15/5000, Train Loss: 0.1503, Val Loss: 0.1389
Epoch 16/5000, Train Loss: 0.1266, Val Loss: 0.1176
Epoch 17/5000, Train Loss: 0.1080, Val Loss: 0.1014
Epoch 18/5000, Train Loss: 0.0941, Val Loss: 0.0895
Epoch 19/5000, Train Loss: 0.0843, Val Loss: 0.0812
Epoch 20/5000, Train 

Exception ignored in: <function _xla_gc_callback at 0x7b22f8943eb0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/jax/_src/lib/__init__.py", line 98, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


Epoch 80/5000, Train Loss: 0.0094, Val Loss: 0.0094
Epoch 81/5000, Train Loss: 0.0092, Val Loss: 0.0093


Exception ignored in: <function _xla_gc_callback at 0x7b22f8943eb0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/jax/_src/lib/__init__.py", line 98, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
# 测试过程
model2.eval()  # 确保模型处于评估模式
test_loss = 0.0
with torch.no_grad():  # 不计算梯度，以节省计算资源和内存
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model2(inputs)
        loss = criterion(outputs, targets)
        test_loss += loss.item() * inputs.size(0)
test_loss /= len(test_loader.dataset)

print(f"Test Loss: {test_loss:.4f}")

Test Loss: 0.0091


In [ ]:
# 定义保存模型的路径，确保路径中的文件夹已经存在
model2_weight_path = '/content/drive/MyDrive/data/model2_final.pth'

# 训练循环结束后保存模型权重
torch.save(model2.state_dict(), model2_weight_path)

In [ ]:
# Load training data (boundary points), residual and interface points from .mat file
# All points are generated in Matlab
main_folder_path = Path('.')
if not os.path.exists(main_folder_path / 'Contour_Plots_NL'):
    os.makedirs(main_folder_path / 'Contour_Plots_NL')

data1 = scipy.io.loadmat(main_folder_path / 'data1.mat')

L_pre=data1['Pred']



# 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




# 使用之前训练数据上 fit 过的 scaler_L 进行变换
L_pre_normalized = scaler_L.transform(L_pre)

# 转换成张量
L_pre_tensor = torch.tensor(L_pre_normalized, dtype=torch.float32)

# 确保模型在正确的设备上
model2.to(device)

# 确保输入数据也在同一设备上
L_pre_tensor = L_pre_tensor.to(device)



# 使用模型进行预测
model2.eval()  # 确保模型处于评估模式
R_pred = model2(L_pre_tensor)



In [ ]:
print(R_pred)

tensor([[-2.3301],
        [-2.2195],
        [-2.1111],
        ...,
        [ 0.5121],
        [ 0.2769],
        [ 0.0174]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [ ]:
#Download the value of R_pred
#Save xy
import numpy as np
from google.colab import files

# 将预测的结果从 GPU 移到 CPU
R_pred_cpu = R_pred.detach().cpu()

# 现在可以安全地将张量转换为 NumPy 数组
R_pred_numpy = R_pred_cpu.numpy()


# 使用scaler_R的inverse_transform方法还原标准化前的数据
R_pred_original = scaler_R.inverse_transform(R_pred_numpy)

np.savetxt('R_pred_original.csv', R_pred_original, delimiter=',')
files.download('R_pred_original.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>